# Lab 10: MLP Model for Time-Series Forecasting

**Student Name:** Malik Awais Bashir  
**Registration Number:** 22JZELE0474  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus  

## Goal of this Lab
* Configure the working environment and import the necessary TensorFlow, machine learning, and supporting libraries.
* Design an MLP (Multi-Layer Perceptron) architecture for processing time-series data.
* Set up the optimizer, model checkpointing, and training callback functions.
* Import training, validation, and testing datasets from CSV files for forecasting tasks.
* Train the MLP model and measure its performance using MAE, MSE, RMSE, R², and explained variance metrics.



## Section 1: Working Directory and Library Imports
This section sets the Lab 10/11 working path and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utilities.


In [40]:
import os
os.chdir(r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11')

In [41]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [42]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [43]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [44]:
model1 = MLP()
model1.summary()

c:\Users\hp\anaconda3\envs\dsp\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_6 (Flatten)             │ (None, 504)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │        16,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,193 (63.25 KB)

 Trainable params: 16,193 (63.25 KB)

 Non-trainable params: 0 (0.00 B)

## Section 2: Model Parameters and MLP Architecture
The following cells define time steps, number of features, and the MLP model structure used for forecasting.


In [45]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [46]:
checkpoints = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [47]:
os.path.exists(JSON_PATH)

False

In [48]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [49]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## Section 3: Checkpoints, Callbacks, and Training Setup
This section configures model checkpoints, training monitor paths, optimizer settings, and compile/training parameters.


In [50]:
import os
path_dataset =r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\hp\anaconda3\envs\dsp\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [51]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.00846719741821289 sec


In [52]:
train_X.shape

(835, 24, 21)

In [53]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/60
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.2260 - mae: 0.2260 - mape: 317.0580
Epoch 1: val_loss improved from None to 0.08063, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0001-loss0.08.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - loss: 0.1469 - mae: 0.1469 - mape: 126.1427 - val_loss: 0.0806 - val_mae: 0.0806 - val_mape: 24.6700
Epoch 2/60
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1073 - mae: 0.1073 - mape: 61.8992
Epoch 2: val_loss improved from 0.08063 to 0.06867, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0002-loss0.07.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - loss: 0.0863 - mae: 0.0863 - mape: 43.2453 - val_loss: 0.0687 - val_mae: 0.0687 - val_mape: 22.0238
Epoch 3/60
22/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0601 - mae: 0.0601 - mape: 28.7167
Epoch 3: val_loss improved from 0.06867 to 0.06042, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0003-loss0.06.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0562 - mae: 0.0562 - mape: 30.3513 - val_loss: 0.0604 - val_mae: 0.0604 - val_mape: 18.8638
Epoch 4/60
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0483 - mae: 0.0483 - mape: 24.4031
Epoch 4: val_loss did not improve from 0.06042
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.0500 - mae: 0.0500 - mape: 25.0209 - val_loss: 0.0770 - val_mae: 0.0770 - val_mape: 25.2012
Epoch 5/60
21/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0472 - mae: 0.0472 - mape: 21.8908 
Epoch 5: val_loss did not improve from 0.06042
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0492 - mae: 0.0492 - mape: 25.7878 - val_loss: 0.0632 - val_mae: 0.0632 - val_mape: 23.9847
Epoch 6/60
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0524 - mae: 0.0524 - mape: 29.5734
Epoch 6: val_loss improved from 0.06042 to 0.04773, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0006-loss0.05.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.0497 - mae: 0.0497 - mape: 25.5729 - val_loss: 0.0477 - val_mae: 0.0477 - val_mape: 16.7338
Epoch 7/60
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0454 - mae: 0.0454 - mape: 22.3412
Epoch 7: val_loss did not improve from 0.04773
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0463 - mae: 0.0463 - mape: 22.2518 - val_loss: 0.0695 - val_mae: 0.0695 - val_mape: 26.0495
Epoch 8/60
21/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0595 - mae: 0.0595 - mape: 23.5444
Epoch 8: val_loss improved from 0.04773 to 0.03974, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0008-loss0.04.h5


...

27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.0305 - mae: 0.0305 - mape: 15.0308 - val_loss: 0.0350 - val_mae: 0.0350 - val_mape: 10.2521
Epoch 43/60
21/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0285 - mae: 0.0285 - mape: 14.6001
Epoch 43: val_loss did not improve from 0.03496
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.0285 - mae: 0.0285 - mape: 14.3338 - val_loss: 0.0356 - val_mae: 0.0356 - val_mape: 10.2120
Epoch 44/60
22/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0267 - mae: 0.0267 - mape: 11.7972
Epoch 44: val_loss did not improve from 0.03496
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0271 - mae: 0.0271 - mape: 13.1893 - val_loss: 0.0420 - val_mae: 0.0420 - val_mape: 14.0079
Epoch 45/60
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0300 - mae: 0.0300 - mape: 15.7376
Epoch 45: val_loss did not improve from 0.03496
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0287 - mae: 0.0287 - mape: 16.6048 - val_loss: 0.0519 - val_mae: 0.0519 - val_mape: 17.1164


27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - loss: 0.0257 - mae: 0.0257 - mape: 13.0823 - val_loss: 0.0335 - val_mae: 0.0335 - val_mape: 9.8594
Epoch 48/60
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0321 - mae: 0.0321 - mape: 15.6634
Epoch 48: val_loss did not improve from 0.03351
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - loss: 0.0311 - mae: 0.0311 - mape: 14.8243 - val_loss: 0.0568 - val_mae: 0.0568 - val_mape: 17.2291
Epoch 49/60
20/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0341 - mae: 0.0341 - mape: 14.0723 
Epoch 49: val_loss did not improve from 0.03351
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.0281 - mae: 0.0281 - mape: 13.1265 - val_loss: 0.0359 - val_mae: 0.0359 - val_mape: 11.4505
Epoch 50/60
20/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0242 - mae: 0.0242 - mape: 10.7658 
Epoch 50: val_loss did not improve from 0.03351
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0272 - mae: 0.0272 - mape: 13.2622 - val_loss: 0.0583 - val_mae: 0.0583 - val_mape: 18.921

27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.0206 - mae: 0.0206 - mape: 11.0456 - val_loss: 0.0331 - val_mae: 0.0331 - val_mape: 10.5581
Epoch 54/60
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0223 - mae: 0.0223 - mape: 11.0614 
Epoch 54: val_loss did not improve from 0.03311
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - loss: 0.0208 - mae: 0.0208 - mape: 10.0196 - val_loss: 0.0415 - val_mae: 0.0415 - val_mape: 12.6297
Epoch 55/60
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0278 - mae: 0.0278 - mape: 14.0430 
Epoch 55: val_loss improved from 0.03311 to 0.03301, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0055-loss0.03.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - loss: 0.0239 - mae: 0.0239 - mape: 11.8372 - val_loss: 0.0330 - val_mae: 0.0330 - val_mape: 10.4343
Epoch 56/60
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0231 - mae: 0.0231 - mape: 11.8863 
Epoch 56: val_loss improved from 0.03301 to 0.03215, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0056-loss0.03.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - loss: 0.0218 - mae: 0.0218 - mape: 11.2110 - val_loss: 0.0321 - val_mae: 0.0321 - val_mape: 8.4635
Epoch 57/60
15/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0218 - mae: 0.0218 - mape: 9.3362 
Epoch 57: val_loss did not improve from 0.03215
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 0.0227 - mae: 0.0227 - mape: 11.0651 - val_loss: 0.0338 - val_mae: 0.0338 - val_mape: 9.9527
Epoch 58/60
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0229 - mae: 0.0229 - mape: 13.7671 
Epoch 58: val_loss did not improve from 0.03215
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 0.0215 - mae: 0.0215 - mape: 11.8168 - val_loss: 0.0356 - val_mae: 0.0356 - val_mape: 10.6467
Epoch 59/60
15/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0196 - mae: 0.0196 - mape: 7.5851  
Epoch 59: val_loss did not improve from 0.03215
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - loss: 0.0224 - mae: 0.0224 - mape: 11.1112 - val_loss: 0.0344 - val_mae: 0.0344 - val_mape: 10.9828


In [56]:
model = load_model(r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0056-loss0.03.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step
Mean Absolute Error (MAE): 1285.53
Median Absolute Error (MedAE): 1279.52
Mean Squared Error (MSE): 2019592.24
Root Mean Squared Error (RMSE): 1421.12
Mean Absolute Percentage Error (MAPE): 8.23 %
Median Absolute Percentage Error (MDAPE): 8.29 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Section 4: Dataset Loading and Forecast Evaluation
The remaining cells load CSV datasets, prepare forecasting windows, run prediction, and evaluate model performance using regression metrics.


# Fine Tuning

In [65]:
checkpoints = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0055-loss0.03.h5'
start_epoch= 56

In [66]:
# construct the callback to save only the best model to disk
EpochCheckpoint1 = ModelCheckpoint(
    checkpoints,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

TrainingMonitor1 = TrainingMonitor(
    FIG_PATH,
    jsonPath=JSON_PATH,
    startAt=start_epoch
)

callbacks = [EpochCheckpoint1, TrainingMonitor1]

model_path = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0055-loss0.03.h5'
# model_path = None   # use this if you want to train from scratch

if model_path is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)

    opt = Adam(learning_rate=1e-3)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

else:
    print("[INFO] loading {}...".format(model_path))
    model = load_model(model_path, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

    print("[INFO] learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0055-loss0.03.h5...
[INFO] learning rate: 9.999999747378752e-05


In [67]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0206 - mae: 0.0206 - mape: 10.2949
Epoch 1: val_loss improved from None to 0.03050, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E2-cp-0001-loss0.03.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step - loss: 0.0176 - mae: 0.0176 - mape: 8.3793 - val_loss: 0.0305 - val_mae: 0.0305 - val_mape: 8.8810
Epoch 2/10
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0160 - mae: 0.0160 - mape: 6.9639 
Epoch 2: val_loss did not improve from 0.03050
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - loss: 0.0156 - mae: 0.0156 - mape: 6.6561 - val_loss: 0.0316 - val_mae: 0.0316 - val_mape: 9.1598
Epoch 3/10
22/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0147 - mae: 0.0147 - mape: 6.0604
Epoch 3: val_loss did not improve from 0.03050
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0150 - mae: 0.0150 - mape: 6.2286 - val_loss: 0.0324 - val_mae: 0.0324 - val_mape: 9.3471
Epoch 4/10
21/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0161 - mae: 0.0161 - mape: 7.3135
Epoch 4: val_loss did not improve from 0.03050
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0157 - mae: 0.0157 - mape: 6.6443 - val_loss: 0.0318 - val_mae: 0.0318 - val_mape: 9.3139
Epoch 5/10
20/27

In [69]:
model = load_model(r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E2-cp-0001-loss0.03.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step
Mean Absolute Error (MAE): 1202.9
Median Absolute Error (MedAE): 1212.43
Mean Squared Error (MSE): 1801139.19
Root Mean Squared Error (RMSE): 1342.07
Mean Absolute Percentage Error (MAPE): 7.71 %
Median Absolute Percentage Error (MDAPE): 7.86 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Conclusion
* In this lab, an MLP (Multi-Layer Perceptron) model was developed for time-series forecasting applications.
* The neural network architecture was designed and configured for predictive analysis.
* Training procedures, callbacks, and monitoring techniques were implemented during model development.
* The forecasting model was trained and tested using time-series datasets.
* Model performance was assessed through various regression evaluation metrics to measure prediction accuracy.
